# XES aggregate results — mfx102101026

Reads the per-run spectra saved by `mfx102101026_static_xes_visualization.ipynb`
and aggregates them across runs.  Re-running the full pipeline is **not** required.

**Workflow:**
1. Run the XSpect pipeline (cell 1 of the visualization notebook) for each new run.
2. Execute the *Save* cell in that notebook to update the HDF5 in `results/`.
3. Re-run **this** notebook to see updated aggregate plots.

Sections:
- **1. Load** — reads HDF5 + `runs.csv`
- **2. Per-run summary table** — shots, signal, hit rate
- **3. Combined spectrum** — photon-weighted sum across all selected runs
- **4. Cumulative buildup** — SNR improvement as runs are added
- **5. Per-run normalized overlay** — consistency check
- **6. IAD / LCF** — requires `runs.csv` with sample composition

In [1]:
import os, csv
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

HERE     = os.path.dirname(os.path.abspath('__file__'))
RESULTS  = os.path.join(HERE, 'results')
CSV_FILE = os.path.join(HERE, 'runs.csv')

# ── locate the most recent results HDF5 ──────────────────────────────────────
h5_files = sorted(
    [f for f in os.listdir(RESULTS) if f.endswith('_static_xes.h5')],
    key=lambda f: os.path.getmtime(os.path.join(RESULTS, f))
)
if not h5_files:
    raise FileNotFoundError(
        f'No *_static_xes.h5 found in {RESULTS}.\n'
        'Run the pipeline and execute the Save cell in '
        'mfx102101026_static_xes_visualization.ipynb first.'
    )
H5_FILE = os.path.join(RESULTS, h5_files[-1])
print(f'Loading results from: {H5_FILE}')

Loading results from: /sdf/data/lcls/ds/mfx/mfx100895324/results/lbgee/XSpect/experiments/mfx102101026/results/run_17_foil_static_xes.h5


## 1. Load results

In [2]:
# ── read HDF5 ─────────────────────────────────────────────────────────────────
with h5py.File(H5_FILE, 'r') as fh:
    ka_energy = fh['kalpha_energy'][:]   # (704,) eV
    kb_energy = fh['kbeta_energy'][:]    # (704,) eV

    run_nums = sorted(
        int(k.split('_')[1]) for k in fh.keys() if k.startswith('run_')
    )
    ka_raw   = {}   # raw (unnormalized) summed spectra
    kb_raw   = {}
    total_shots = {}

    for rn in run_nums:
        g = fh[f'run_{rn}']
        ka_raw[rn]      = g['kalpha'][:]
        kb_raw[rn]      = g['kbeta'][:]
        total_shots[rn] = int(g.attrs.get('total_shots', 0))

# ── read runs.csv ─────────────────────────────────────────────────────────────
sample_label     = {}
reduced_fraction = {}
role             = {}

if os.path.exists(CSV_FILE):
    with open(CSV_FILE) as fh:
        for row in csv.DictReader(fh):
            rn = int(row['run'])
            sample_label[rn] = row.get('sample_label', '')
            role[rn]         = row.get('role', '')
            try:
                reduced_fraction[rn] = float(row['reduced_fraction'])
            except (ValueError, KeyError):
                pass

FOIL_RUNS = [rn for rn in run_nums if role.get(rn) == 'excluded']
REF_RUNS  = [rn for rn in run_nums
             if reduced_fraction.get(rn) == 0.0 and rn not in FOIL_RUNS]

print(f'Runs loaded:  {run_nums}')
print(f'Reference:    {REF_RUNS}')
print(f'Foil/excl:    {FOIL_RUNS}')
print(f'Kα energy range: {ka_energy.min():.1f} – {ka_energy.max():.1f} eV  ({len(ka_energy)} px)')

KeyError: "Unable to synchronously open object (object 'kbeta_energy' doesn't exist)"

## 2. Per-run summary

In [ ]:
def area_norm(y):
    y = np.asarray(y, dtype=float)
    s = np.nansum(y)
    return y / s if s > 0 else y

header = f'{"run":>4}  {"sample":12}  {"shots":>7}  {"Kα sum":>10}  {"Kβ sum":>10}  {"Kα/shot":>8}'
print(header)
print('-' * len(header))
for rn in run_nums:
    ka_s  = np.nansum(ka_raw[rn])
    kb_s  = np.nansum(kb_raw[rn])
    n     = total_shots[rn]
    lbl   = sample_label.get(rn, '?')[:12]
    print(f'{rn:>4}  {lbl:12}  {n:>7}  {ka_s:>10.0f}  {kb_s:>10.0f}  {ka_s/n if n else 0:>8.3f}')

print('-' * len(header))
ka_total = np.nansum([ka_raw[rn] for rn in run_nums], axis=0)
kb_total = np.nansum([kb_raw[rn] for rn in run_nums], axis=0)
n_total  = sum(total_shots.values())
print(f'{"ALL":>4}  {"":12}  {n_total:>7}  '
      f'{np.nansum(ka_total):>10.0f}  {np.nansum(kb_total):>10.0f}  '
      f'{np.nansum(ka_total)/n_total if n_total else 0:>8.3f}')

## 3. Combined spectrum — all runs summed

Raw photon-weighted sum (left) and area-normalized (right) with tabulated
reference positions overlaid.

In [ ]:
# Select runs to include in the combined spectrum
# Default: all runs except foil/excluded.  Override by setting include_runs.
include_runs = [rn for rn in run_nums if rn not in FOIL_RUNS]

ka_combined = np.nansum([ka_raw[rn] for rn in include_runs], axis=0)
kb_combined = np.nansum([kb_raw[rn] for rn in include_runs], axis=0)
n_combined  = sum(total_shots[rn] for rn in include_runs)

fig, axes = plt.subplots(2, 2, figsize=(15, 8))

for row, (energy, raw, norm_spec, title, ref_lines) in enumerate([
    (ka_energy, ka_combined, area_norm(ka_combined),
     'Fe Kα  (combined)', [6404, 6391]),
    (kb_energy, kb_combined, area_norm(kb_combined),
     'Fe Kβ  (combined)', [7058]),
]):
    ax_raw  = axes[row][0]
    ax_norm = axes[row][1]

    ax_raw.plot(energy, raw, color='C0' if row == 0 else 'C3', lw=1.5)
    ax_raw.set_ylabel('Summed ADU')
    ax_raw.set_title(f'{title} — raw sum ({len(include_runs)} runs, {n_combined} shots)')

    ax_norm.plot(energy, norm_spec, color='C0' if row == 0 else 'C3', lw=1.5)
    ax_norm.set_ylabel('Area-normalised intensity')
    ax_norm.set_title(f'{title} — area-normalised')

    for ax in (ax_raw, ax_norm):
        for e in ref_lines:
            ax.axvline(e, color='gray', ls=':', lw=0.8)
        ax.set_xlabel('Emission energy (eV)')
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Combined: {len(include_runs)} runs, {n_combined} shots')
print(f'  Kα total ADU: {np.nansum(ka_combined):.0f}   '
      f'peak position: {ka_energy[np.argmax(ka_combined)]:.1f} eV')
print(f'  Kβ total ADU: {np.nansum(kb_combined):.0f}   '
      f'peak position: {kb_energy[np.argmax(kb_combined)]:.1f} eV')

## 4. Cumulative signal buildup

Shows how the summed spectrum evolves as runs are added in order.
The right panel tracks peak-region SNR vs cumulative shot count —
useful for estimating how many more runs are needed.

In [ ]:
cmap   = cm.viridis
n_inc  = len(include_runs)
cnorm  = mcolors.Normalize(1, n_inc)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

cumulative_shots = []
ka_snr_list = []
kb_snr_list = []
ka_cum = np.zeros_like(ka_combined)
kb_cum = np.zeros_like(kb_combined)
n_cum  = 0

for i, rn in enumerate(include_runs):
    ka_cum += ka_raw[rn]
    kb_cum += kb_raw[rn]
    n_cum  += total_shots[rn]
    colour  = cmap(cnorm(i + 1))
    lbl     = f'run {rn}' if i in (0, n_inc - 1) else None

    axes[0].plot(ka_energy, area_norm(ka_cum), color=colour, lw=0.9, alpha=0.8, label=lbl)
    axes[1].plot(kb_energy, area_norm(kb_cum), color=colour, lw=0.9, alpha=0.8, label=lbl)

    # SNR = peak / noise-floor (std of a flat off-peak region)
    def snr(spec, energy, peak_e, win=5.0, noise_lo=None, noise_hi=None):
        sig  = spec[(energy >= peak_e - win) & (energy <= peak_e + win)].sum()
        if noise_lo is not None and noise_hi is not None:
            noise_region = spec[(energy >= noise_lo) & (energy <= noise_hi)]
        else:
            noise_region = spec  # fallback
        noise = np.std(noise_region)
        return sig / noise if noise > 0 else 0

    cumulative_shots.append(n_cum)
    ka_snr_list.append(snr(ka_cum, ka_energy, 6404,
                           noise_lo=ka_energy.min(), noise_hi=ka_energy.min()+20))
    kb_snr_list.append(snr(kb_cum, kb_energy, 7058,
                           noise_lo=kb_energy.min(), noise_hi=kb_energy.min()+20))

for ax, elines, ttl in [
    (axes[0], [6404, 6391], 'Fe Kα — cumulative'),
    (axes[1], [7058],       'Fe Kβ — cumulative'),
]:
    for e in elines:
        ax.axvline(e, color='gray', ls=':', lw=0.8)
    ax.set_xlabel('Emission energy (eV)')
    ax.set_ylabel('Area-norm. intensity')
    ax.set_title(ttl)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=cnorm)
sm.set_array([])
fig.colorbar(sm, ax=axes[1], label='Runs added', fraction=0.046)

ax = axes[2]
ax.plot(cumulative_shots, ka_snr_list, 'o-', color='C3', label='Kα SNR')
ax.plot(cumulative_shots, kb_snr_list, 's-', color='C0', label='Kβ SNR')
# √N scaling guide
if cumulative_shots:
    n0 = cumulative_shots[0]
    snr0_ka = ka_snr_list[0] if ka_snr_list[0] > 0 else 1
    ns_guide = np.array(cumulative_shots)
    ax.plot(ns_guide, snr0_ka * np.sqrt(ns_guide / n0),
            'k--', lw=0.8, alpha=0.5, label='√N scaling')
ax.set_xlabel('Cumulative shots')
ax.set_ylabel('Peak SNR (peak sum / baseline std)')
ax.set_title('SNR buildup')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Per-run normalized overlay

Each run area-normalized independently.  Outliers (bad runs, flow issues,
alignment drifts) show up as spectral shape deviations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for rn in include_runs:
    c   = cmap(cnorm(include_runs.index(rn) + 1))
    lbl = f'run {rn}' + (f'  ({sample_label[rn]})' if rn in sample_label else '')
    axes[0].plot(ka_energy, area_norm(ka_raw[rn]), color=c, lw=1.0, alpha=0.85, label=lbl)
    axes[1].plot(kb_energy, area_norm(kb_raw[rn]), color=c, lw=1.0, alpha=0.85, label=lbl)

# Combined on top
axes[0].plot(ka_energy, area_norm(ka_combined), 'k-', lw=2, alpha=0.5, label='combined')
axes[1].plot(kb_energy, area_norm(kb_combined), 'k-', lw=2, alpha=0.5, label='combined')

for ax, elines, ttl in [
    (axes[0], [6404, 6391], 'Fe Kα — per-run overlay'),
    (axes[1], [7058],       'Fe Kβ — per-run overlay'),
]:
    for e in elines:
        ax.axvline(e, color='gray', ls=':', lw=0.8)
    ax.set_xlabel('Emission energy (eV)')
    ax.set_ylabel('Area-norm. intensity')
    ax.set_title(ttl)
    if len(include_runs) <= 12:
        ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. IAD and LCF

> **Requires** `runs.csv` with `reduced_fraction` filled in for at least one
> reference run (`reduced_fraction = 0.0`) and one reduced end-member
> (`reduced_fraction = 1.0`).  Cells skip gracefully until that information
> is available.

In [ ]:
chem_runs = [rn for rn in run_nums
             if rn not in FOIL_RUNS and rn in reduced_fraction]

if not REF_RUNS or not chem_runs:
    print('Skipping IAD — fill in reduced_fraction in runs.csv first.')
else:
    def combined_ref(raw_dict, ref_runs):
        return area_norm(np.nansum([raw_dict[r] for r in ref_runs], axis=0))

    ref_ka = combined_ref(ka_raw, REF_RUNS)
    ref_kb = combined_ref(kb_raw, REF_RUNS)

    iad_ka = {rn: float(np.nansum(np.abs(area_norm(ka_raw[rn]) - ref_ka))) for rn in chem_runs}
    iad_kb = {rn: float(np.nansum(np.abs(area_norm(kb_raw[rn]) - ref_kb))) for rn in chem_runs}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, iad, lbl, col in [
        (axes[0], iad_ka, 'Kα', 'C3'),
        (axes[1], iad_kb, 'Kβ', 'C0'),
    ]:
        xs = [reduced_fraction[rn] for rn in chem_runs]
        ys = [100 * iad[rn] for rn in chem_runs]
        ax.scatter(xs, ys, c=col, s=50, zorder=3)
        for rn in chem_runs:
            ax.annotate(str(rn),
                        (reduced_fraction[rn], 100 * iad[rn]),
                        fontsize=7, xytext=(3, 3), textcoords='offset points')
        ax.set_xlabel('Nominal Fe(II) fraction')
        ax.set_ylabel('IAD × 100')
        ax.set_title(f'{lbl}: IAD vs composition')
        ax.grid(alpha=0.3)
    fig.suptitle(f'IAD reference = combined runs {REF_RUNS}')
    plt.tight_layout()
    plt.show()

In [ ]:
RED_RUNS = [rn for rn in run_nums
            if reduced_fraction.get(rn) == 1.0 and rn not in FOIL_RUNS]

if not REF_RUNS or not RED_RUNS or not chem_runs:
    print('Skipping LCF — need both reduced_fraction=0.0 and =1.0 runs in runs.csv.')
else:
    def lcf(raw_dict, ref_runs, red_runs):
        ref  = combined_ref(raw_dict, ref_runs)
        red  = combined_ref(raw_dict, red_runs)
        diff = red - ref
        return {
            rn: float(np.dot(diff, area_norm(raw_dict[rn]) - ref) / np.dot(diff, diff))
            for rn in chem_runs
        }

    f_ka = lcf(ka_raw, REF_RUNS, RED_RUNS)
    f_kb = lcf(kb_raw, REF_RUNS, RED_RUNS)

    fig, ax = plt.subplots(figsize=(7, 7))
    for fv, lbl, col in [(f_ka, 'Kα', 'C3'), (f_kb, 'Kβ', 'C0')]:
        ax.scatter(
            [reduced_fraction[rn] for rn in chem_runs],
            [fv[rn] for rn in chem_runs],
            c=col, s=50, label=lbl, zorder=3
        )
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='ideal')
    ax.set_xlabel('Nominal Fe(II) fraction')
    ax.set_ylabel('LCF fitted fraction')
    ax.set_title('Fraction reduced: LCF vs nominal')
    ax.set_aspect('equal')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()